## 1. Read JSON

In [8]:
import json
import pandas as pd

dir = r"C:\Users\User\Documents\catalogue_parser\src\full_results.json"

with open(dir, "r", encoding="utf-8") as f:
    data = json.load(f)

# extract products list
products = data["products"]

# convert to dataframe
df = pd.json_normalize(products)

JSONDecodeError: Expecting ',' delimiter: line 29465 column 2 (char 1094997)

* Pages are always +3

In [ ]:
df['page'] = df['page']+3

In [ ]:
df.columns

Index(['page', 'model', 'price', 'dimensions', 'seater', 'materials.frame',
       'materials.armrest', 'materials.table_top', 'colors.cushion',
       'colors.frame',
       ...
       'materials.features', 'other.packing', 'other.fabric_color_count',
       'other.filling', 'other.weight', 'materials.roof', 'colors.net',
       'other.variants', 'other.shade_size', 'other.product_id_numbers'],
      dtype='str', length=130)

## 2. remove known bad models

In [ ]:
df = df.drop(df[df["model"] == "MOD#M015"].index)

In [ ]:
# column_snapshot = pd.DataFrame({
#     "column": df.columns,
#     "dtype": [str(df[c].dtype) for c in df.columns],
#     "non_null": [df[c].notna().sum() for c in df.columns]
# })
# column_snapshot.to_excel("column_audit.xlsx", index=False)

## 3. Clean up price

In [15]:
df["price"] = df["price"].fillna(df["price_2"], inplace=True)
df.drop(columns=['price_2'],inplace=True)

df["Top Material"] = df["Top Material"].fillna(df["Top Material_2"], inplace=True)
df.drop(columns=['Top Material_2'],inplace=True)

KeyError: 'price_2'

In [ ]:
df["price"] = df["price"].astype(str).str[1:-1]

In [ ]:
df['price'].head(3)

0    675
1    600
2    550
Name: price, dtype: str

## 4. Clean up seater

In [ ]:
df["seater"] = df["seater"].str.replace("set", "", case=False)

In [ ]:
df['seater'].head(3)

0      corner + 1+ Table
1        Corner  + table
2    Corner  + 1 + Table
Name: seater, dtype: str

## 5. Clean up Supplier code

In [ ]:
df["model"] = df["model"].str.replace("MOD#", "", case=False)

In [ ]:
df['model']

0     9519/5
1      61337
2       3019
3       7088
4    089/203
5    089/203
Name: model, dtype: str

## 6. Clean up Dimensions

In [ ]:
import re

def format_dimensions_block(raw: str, seater: str = "") -> str:
    if not isinstance(raw, str):
        return ""

    parts = re.split(r',\s*', raw)  # split: Set, Seat, Table...

    output = []

    for part in parts:
        part = part.strip()

        # extract label (before :)
        if ":" in part:
            label, values = part.split(":", 1)
        else:
            label, values = "Dimensions", part

        label = label.strip()

        nums = re.findall(r'\d+(?:[./]\d+)?', values)

        if len(nums) >= 3:
            dim = f"Width: {nums[0]}cm Depth: {nums[1]}cm Height: {nums[2]}cm"
        elif len(nums) == 2:
            dim = f"Width: {nums[0]}cm Depth: {nums[1]}cm"
        else:
            continue

        # special naming
        if label.lower().startswith("set"):
            line = f"{label}: {dim}"
        elif "seat" in label.lower():
            line = f"{label} ({seater}): {dim}" if seater else f"{label}: {dim}"
        else:
            line = f"{label} — Dimensions: {dim}"

        output.append(line)

    return " <br>\n".join(output)

df["description"] = df.apply(
    lambda r: format_dimensions_block(r["dimensions"], r["seater"]),
    axis=1
)
df.drop(columns=['dimensions'], inplace=True)

## 7. Create variable, variation

In [ ]:
df["Type"] = "simple"

mask = df.duplicated(subset=["model", "description"], keep=False)

df.loc[mask, "Type"] = "variation"

In [ ]:
import pandas as pd

def add_variable_parents(df: pd.DataFrame) -> pd.DataFrame:
    df = df.fillna("").copy()

    output_rows = []

    # group by MODEL + dESCRIPTION (your real identity key)
    grouped = df.groupby(["model", "description"], sort=False)

    for (model, desc), group in grouped:
        rows = group.to_dict("records")

        # ── CASE 1: single product → simple ─────────────────────────────
        if len(rows) == 1:
            r = rows[0]
            r["Type"] = "simple"
            output_rows.append(r)
            continue

        # ── CASE 2: multiple rows → variable + variations ──────────────

        # ---- PARENT (variable) ----
        parent = rows[0].copy()
        parent["Type"] = "variable"
        parent["model"] = model
        parent["description"] = desc

        # optional: clean name choice
        names = [r.get("Name", "") for r in rows if r.get("Name")]
        parent["Name"] = min(names, key=len) if names else model

        # clear ONLY attribute columns (keep structure intact)
        for col in df.columns:
            if "Attribute" in col:
                parent[col] = ""

        output_rows.append(parent)

        # ---- VARIATIONS (KEEP EVERYTHING EXACT) ----
        for r in rows:
            r = r.copy()
            r["Type"] = "variation"
            output_rows.append(r)

    return pd.DataFrame(output_rows, columns=df.columns)

df = add_variable_parents(df)

In [ ]:
df.loc[df["Type"] == "variation", "description"] = ""

In [ ]:
df

,page,model,price,seater,Furniture Material,Cushion Color,Frame Color,other.type,Top Material,Armrest Material,Foldable Arm,Textile Color,Aluminum Color,description,Type
0,90,9519/5,675,corner + 1+ Table,Aluminum frame,"[green, gray, red, blue]",white,,,,,,,Set: Width: 242cm Depth: 242cm Height: 65cm <b...,simple
1,90,61337,600,Corner + table,Aluminum frame,gray,white,,,,,,,Set: Width: 306cm Depth: 244cm Height: 74cm <b...,simple
2,91,3019,550,Corner + 1 + Table,Aluminum frame,gray,white,,Glass,,,,,Set: Width: 243cm Depth: 186cm Height: 68cm <b...,simple
3,91,7088,600,Corner + table,Aluminum frame,gray,white,,Full polywood,Full polywood,True,,,Set: Width: 246cm Depth: 246cm Height: 80cm <b...,simple
4,177,089/203,600,Bar table + 4 Bar chairs,Aluminum,,,,Polywood,,,Light gray,White,Dimensions — Dimensions: Width: 130cm Depth: 7...,variable
5,177,089/203,600,Bar table + 4 Bar chairs,Aluminum,,,,Polywood,,,Light gray,White,,variation
6,177,089/203,600,Bar table + 4 Bar chairs,Aluminum,,,,Polywood,,,Dark gray,Dark gray,,variation


## 8. Map SKU, Name and Category by page

In [ ]:
map = pd.read_excel(r"C:\Users\User\Documents\catalogue_parser\src\mapping.xlsx", sheet_name="naming")

map = map[['SKU', 'Pages','Name','Categories']]

In [ ]:
expanded = []

def parse_range(r):
    r = str(r)
    if "-" in r:
        start, end = r.split("-")
        return int(start), int(end)
    return int(r), int(r)

for _, row in map.iterrows():
    start, end = parse_range(row["Pages"])
    for p in range(start, end + 1):
        expanded.append((p, row["SKU"], row["Name"], row["Categories"]))

exp_df = pd.DataFrame(expanded, columns=["page", "SKU", "Name", "Categories"])

In [ ]:
mask_main = df["Type"].isin(["simple", "variable"])
mask_variation = df["Type"] == "variation"

df_main = df[mask_main].copy()
df_other = df[~mask_main].copy()

*  Map SKU

In [ ]:
df_main = df_main.merge(exp_df[["page", "SKU"]], on="page", how="left")

*  Map Categories

In [ ]:
df_main = df_main.merge(exp_df[["page", "Categories"]], on="page", how="left")

In [ ]:
df = pd.concat([df_main, df_other], ignore_index=True)

* Map Name

In [ ]:
df = df.merge(exp_df[["page", "Name"]], on="page", how="left")

## Iterate SKUs per variable and simple

In [ ]:
df["sku_base"] = df["SKU"].str.replace(r"-\d+$", "", regex=True)

In [ ]:
current_parent_key = None
parent_counters = {}
skus = []

In [ ]:
for i, row in df.iterrows():

    sku_base = row["SKU"] if pd.notna(row["SKU"]) else None
    row_type = row["Type"]

    # NEW VARIABLE → create new parent SKU
    if row_type == "variable":
        parent_counters[sku_base] = parent_counters.get(sku_base, 0) + 1
        seq = parent_counters[sku_base]

        current_parent_key = f"{sku_base}-{seq:04d}"
        skus.append(current_parent_key)

    # VARIATION → copy last variable
    elif row_type == "variation":
        skus.append(current_parent_key)

    # SIMPLE → independent SKU
    else:
        parent_counters[sku_base] = parent_counters.get(sku_base, 0) + 1
        seq = parent_counters[sku_base]
        skus.append(f"{sku_base}-{seq:04d}")

df["SKU"] = skus

In [ ]:
df.drop(columns=["sku_base"], inplace=True)

df["SKU"] = df["SKU"].str.replace(r"-0000-", "-", regex=True)

In [ ]:
df['SKU']

0    OUT-SFST-FUR-0001
1    OUT-SFST-FUR-0002
2    OUT-SFST-FUR-0003
3    OUT-SFST-FUR-0004
4    OUT-BRST-FUR-0001
5    OUT-BRST-FUR-0001
6    OUT-BRST-FUR-0001
Name: SKU, dtype: str